# 📸 Урок 16 — Твой проект «Классификатор моих фото»

Сегодня ты обучишь классификатор СВОИХ фото и сделаешь сайт, ссылку на который можно отправить родителям.

> ★ **Оценивается (10 баллов):** свои данные (2) · аугментация (2) · transfer learning (3) · Gradio + ссылка (3).

## Шаг 1 · Выбери категории и сними фото ✍️
Выбери 2–3 понятные категории и сними по 15–20 фото каждой (разный фон и ракурс!).

*Мои категории:* …

Загрузи фото в папки (одна папка = один класс):
```
data/
  класс1/  ...
  класс2/  ...
```

## Как загрузить фото — выбери ОДИН путь

- **Путь А** — есть свои фото → запусти клетку А, нажми кнопку, выбери все файлы сразу.
- **Путь Б** — фото нет или не успел → запусти клетку Б, возьмём запасной датасет (кошки/собаки).

Имена файлов для Пути А начинай с категории и `_`: `cat_1.jpg`, `dog_1.jpg` — по слову до `_` определим класс. Делай только ОДИН путь, потом иди дальше.

In [ ]:
# ПУТЬ А — свои фото через кнопку загрузки
from google.colab import files
import os, shutil

if os.path.exists("data"): shutil.rmtree("data")   # чистим, если запускаешь второй раз
os.makedirs("data", exist_ok=True)

print("Нажми кнопку и выбери ВСЕ свои фото (имена вида cat_1.jpg, dog_1.jpg):")
uploaded = files.upload()

for fname in uploaded.keys():
    category = fname.split("_")[0]              # cat_1.jpg -> "cat"
    folder = os.path.join("data", category)
    os.makedirs(folder, exist_ok=True)
    shutil.move(fname, os.path.join(folder, fname))

print("Готово! Категории:", os.listdir("data"))
for c in os.listdir("data"):
    print(" ", c, "—", len(os.listdir(os.path.join("data", c))), "фото")

In [ ]:
# ПУТЬ Б — запасной датасет кошки/собаки (если своих фото нет)
import tensorflow as tf, os, shutil

url = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"
path = tf.keras.utils.get_file("cats_and_dogs.zip", origin=url, extract=True)
src = os.path.join(os.path.dirname(path), "cats_and_dogs_filtered", "train")

if os.path.exists("data"): shutil.rmtree("data")
for cls_src, cls_dst in [("cats", "cat"), ("dogs", "dog")]:
    dst = os.path.join("data", cls_dst); os.makedirs(dst, exist_ok=True)
    for im in sorted(os.listdir(os.path.join(src, cls_src)))[:30]:
        shutil.copy(os.path.join(src, cls_src, im), os.path.join(dst, im))

print("Запасной датасет готов. Категории:", os.listdir("data"))
for c in os.listdir("data"):
    print(" ", c, "—", len(os.listdir(os.path.join("data", c))), "фото")

## Шаг 2 · Готовим данные (train/val) + аугментация
Делим фото на обучающую и проверочную выборки. Аугментация из одного фото делает много вариантов — модель учится лучше, а по val видно переобучение.

In [ ]:
import tensorflow as tf
from tensorflow import keras

# делим фото на обучающую (80%) и проверочную (20%) выборки
train_ds = keras.utils.image_dataset_from_directory(
    'data', validation_split=0.2, subset='training', seed=42,
    image_size=(160, 160), batch_size=8)
val_ds = keras.utils.image_dataset_from_directory(
    'data', validation_split=0.2, subset='validation', seed=42,
    image_size=(160, 160), batch_size=8)

class_names = train_ds.class_names
print('Категории:', class_names)

augment = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),   # отражение
    keras.layers.RandomRotation(0.1),          # поворот
    keras.layers.RandomZoom(0.1),              # приближение
])

## Шаг 3 · Обучаем на готовой сети (transfer learning)
Берём MobileNet (уже видел миллионы картинок) и доучиваем под свои классы.

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(160,160,3), include_top=False, weights='imagenet')
base.trainable = False
model = keras.Sequential([
    augment,
    keras.layers.Rescaling(1./127.5, offset=-1),
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(len(class_names), activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_ds, validation_data=val_ds, epochs=8)

## Шаг 4 · Сделай сайт через Gradio 🌐
Получишь публичную ссылку — открой её и загрузи новое фото!

In [ ]:
!pip install gradio -q
import gradio as gr, tensorflow as tf, numpy as np
def predict(img):
    img = np.array(img)[..., :3]                    # PNG с прозрачностью -> RGB (3 канала)
    x = tf.image.resize(img, (160,160))[None, ...]
    p = model.predict(x)[0]
    return {class_names[i]: float(p[i]) for i in range(len(class_names))}
gr.Interface(fn=predict, inputs=gr.Image(), outputs=gr.Label(num_top_classes=len(class_names)),
             title='Классификатор моих фото').launch(share=True)   # share=True -> публичная ссылка

✍️ **Ответь.** Где твоя модель путается? Помогла ли аугментация?

*Ответ:* …

## ✅ Проверь себя
1. Зачем аугментация?
2. Почему transfer learning работает даже на 20 фото?
3. Что делает `share=True` в Gradio?

<details><summary>Ответы</summary>

1. Из немногих фото делает много вариантов — больше данных для обучения.
2. Готовая сеть уже умеет узнавать узоры; мы доучиваем только 'голову'.
3. Даёт публичную ссылку на твой веб-интерфейс.
</details>

---
### 🏁 Чек-лист перед сдачей
- [ ] 2–3 категории, по 15–20 фото
- [ ] Применена аугментация
- [ ] Модель обучена (transfer learning) и работает на новых фото
- [ ] Работает Gradio-интерфейс с публичной ссылкой
- [ ] Записано, где модель ошибается

🎉 Отправь ссылку родителям — пусть проверят твой классификатор!